In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train_data = pd.read_csv("./data/bike-sharing-demand/train.csv")
test_data = pd.read_csv("./data/bike-sharing-demand/test.csv")

### 1번 (기계학습): 자전거 대여 수요 예측 및 분류 (40점)

1.  **데이터 전처리:** `datetime` 컬럼을 활용하여 연, 월, 일, 시간, 요일 변수를 추출하고, 결측치 여부를 확인하여 적절한 처리 방안을 제시하시오.

In [2]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   datetime    10886 non-null  object 
 1   season      10886 non-null  int64  
 2   holiday     10886 non-null  int64  
 3   workingday  10886 non-null  int64  
 4   weather     10886 non-null  int64  
 5   temp        10886 non-null  float64
 6   atemp       10886 non-null  float64
 7   humidity    10886 non-null  int64  
 8   windspeed   10886 non-null  float64
 9   casual      10886 non-null  int64  
 10  registered  10886 non-null  int64  
 11  count       10886 non-null  int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 1020.7+ KB


In [5]:
train_data["datetime"]

0        2011-01-01 00:00:00
1        2011-01-01 01:00:00
2        2011-01-01 02:00:00
3        2011-01-01 03:00:00
4        2011-01-01 04:00:00
                ...         
10881    2012-12-19 19:00:00
10882    2012-12-19 20:00:00
10883    2012-12-19 21:00:00
10884    2012-12-19 22:00:00
10885    2012-12-19 23:00:00
Name: datetime, Length: 10886, dtype: object

In [8]:
train_data["datetime"] = pd.to_datetime(train_data["datetime"], format="%Y-%m-%d %H:%M:%S")

In [9]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   datetime    10886 non-null  datetime64[ns]
 1   season      10886 non-null  int64         
 2   holiday     10886 non-null  int64         
 3   workingday  10886 non-null  int64         
 4   weather     10886 non-null  int64         
 5   temp        10886 non-null  float64       
 6   atemp       10886 non-null  float64       
 7   humidity    10886 non-null  int64         
 8   windspeed   10886 non-null  float64       
 9   casual      10886 non-null  int64         
 10  registered  10886 non-null  int64         
 11  count       10886 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(8)
memory usage: 1020.7 KB


In [12]:
train_data["year"] = train_data["datetime"].dt.year
train_data["month"] = train_data["datetime"].dt.month
train_data["day"] = train_data["datetime"].dt.day
train_data["time"] = train_data["datetime"].dt.time
train_data["weekday"] = train_data["datetime"].dt.weekday

In [16]:
train_data.isna().sum()

datetime      0
season        0
holiday       0
workingday    0
weather       0
temp          0
atemp         0
humidity      0
windspeed     0
casual        0
registered    0
count         0
year          0
month         0
day           0
time          0
weekday       0
dtype: int64

2.  **변수 생성:** `count`(대여수) 변수를 바탕으로, 상위 25% 이상인 경우 'High'(1), 그렇지 않은 경우 'Low'(0)인 이진 분류용 종속변수 `demand_level`을 생성하시오.

In [18]:
train_data["count"].describe()
train_data["count"]>284

count    10886.000000
mean       191.574132
std        181.144454
min          1.000000
25%         42.000000
50%        145.000000
75%        284.000000
max        977.000000
Name: count, dtype: float64

In [20]:
train_data["demand_level"] = train_data["count"] >= train_data["count"].quantile(0.75)

In [22]:
sum(train_data["demand_level"])

2724

In [23]:
train_data["demand_level"].count()

10886

3.  **EDA 및 시각화:** 수치형 변수(기온, 습도 등)와 범주형 변수(계절, 날씨 등)를 분류하고, `demand_level`과의 관계를 나타내는 시각화를 3개 이상 수행한 뒤 그 결과를 해석하시오.

4.  **변수 선택:** `demand_level`과 상관관계가 높은 수치형 변수 2개와 범주형 변수 2개를 선정하고, 다중공선성(VIF) 문제를 검토하시오.

5.  **모델링:** 단일 알고리즘(예: Decision Tree)과 앙상블 알고리즘(예: Random Forest 또는 XGBoost)을 각각 학습시키고, 하이퍼파라미터 튜닝 과정을 설명하시오.

6.  **평가:** Confusion Matrix와 F1-score를 사용하여 두 모델의 성능을 비교하고, 최종 모델을 선정한 사유를 서술하시오.

### 2번 (기계학습): 파생 변수 및 시계열 전처리 (20점)

**[전처리 가이드]**
* 주어진 데이터에는 `temp`(실제 기온)와 `atemp`(체감 기온)가 포함되어 있습니다.
* 두 변수의 차이(`diff_temp`)를 계산하여 새로운 변수를 생성하고, 이 변수가 대여량 예측에 유의미한지 산점도를 통해 확인하시오.
* 시간대별로 데이터가 정렬되어 있지 않다면, 시간 순서대로 정렬하고 3시간 단위의 이동평균(Moving Average) 대여량을 계산하여 파생 변수로 추가하시오.

### 3번 (통계분석): 시계열 분해 및 이상치 탐지 (20점)

1.  **시계열 분해:** 전체 대여량(`count`)에 대해 STL 분해를 수행하시오. (주기는 일일 단위인 24로 설정)

2.  **강도 계산:** 분해된 성분을 바탕으로 추세 강도(Trend Strength)와 계절성 강도(Seasonal Strength)를 계산하고, 이 데이터가 계절적 영향을 얼마나 강하게 받는지 해석하시오.

3.  **이상치 판별:** STL 분해 후 발생한 잔차(Remainder)에 대하여 다음 두 가지 기준을 적용하시오.
    * **Z-score:** 2.5 이상인 지점
    * **IQR:** $Q3 + 1.5 \times IQR$ 또는 $Q1 - 1.5 \times IQR$을 벗어나는 지점
    * 두 기준 모두 만족하는 날짜를 추출하고, 해당 날짜의 날씨(Weather) 변수와 연관 지어 이상치 발생 원인을 추론하시오.

### 4번 (통계분석): 가설 검정 (20점)

1.  **평균 차이 검정:** '평일(workingday=1)'과 '주말(workingday=0)'의 평균 자전거 대여량에 차이가 있는지 유의수준 0.05에서 검정하시오. (단, 정규성과 등분산성 가정을 먼저 확인하고 적절한 검정 방법을 선택할 것)

2.  **독립성 검정:** '날씨 상태(weather)'와 '계절(season)' 사이에 연관성이 있는지 카이제곱 검정을 통해 확인하고 결과를 해석하시오.

3.  **회귀 분석:** 기온(`temp`)과 습도(`humidity`)가 대여량(`count`)에 미치는 영향을 다중회귀분석을 통해 확인하고, 회귀 계수의 유의성을 검정하시오.